In [2]:
import os
import numpy as np
import pandas as pd
import librosa

# preprocessing
def extract_mfcc_features(file_path):
    signal, sample_rate = librosa.load(
        file_path,
        sr=None
    )
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sample_rate,
        n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    features = np.concatenate([
        mfcc_mean,
        mfcc_std
    ])
    return features

train_path = r"C:\Users\DESLA\Desktop\ASV_2017\baseline_CM\baseline_CM\ASVspoof2017_V2_train"

data = []
for filename in os.listdir(train_path):
    if filename.lower().endswith(".wav"):
        file_path = os.path.join(
            train_path,
            filename
        )
        features = extract_mfcc_features(
            file_path
        )
        data.append(
            [filename] + features.tolist()
        )

print("Number of audio files:", len(data))

feature_names = []
for i in range(1, 14):
    feature_names.append(f"MFCC_{i}_mean")
for i in range(1, 14):
    feature_names.append(f"MFCC_{i}_std")

columns = ["filename"] + feature_names

df = pd.DataFrame(
    data,
    columns=columns
)

protocol_path = r"C:\Users\DESLA\Desktop\ASV_2017\baseline_CM\baseline_CM\protocol_V2\ASVspoof2017_V2_train.trn.txt"

protocol_data = []
with open(protocol_path, "r") as file:
    for line in file:
        parts = line.strip().split()
        filename = parts[0]
        label = parts[1]
        protocol_data.append([filename, label])

protocol_df = pd.DataFrame(
    protocol_data,
    columns=["filename", "label"]
)

df = df.merge(
    protocol_df,
    on="filename",
    how="inner"
)

X = df[feature_names].to_numpy()
Y = df["label"]

def standardize(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    X_scaled = (X - mean) / std
    return X_scaled, mean, std

X_scaled, mean, std = standardize(X)

def label_encode(y):
    labels = []
    encoded = []
    for value in y:
        if value not in labels:
            labels.append(value)
    mapping = {}
    for i in range(len(labels)):
        mapping[labels[i]] = i
    for value in y:
        encoded.append(mapping[value])
    return encoded, mapping

Y_encoded, label_mapping = label_encode(Y)

def minkowski_distance(X, Y, p):
    total = 0
    for i in range(len(X)):
        total += abs(X[i] - Y[i]) ** p
    distance = total ** (1 / p)
    return distance

def calculate_all_distances(test_pattern, X_train, p):
    distances = []
    for i in range(len(X_train)): 
        distance = minkowski_distance(
            test_pattern,
            X_train[i],
            p
        )
        distances.append(distance)
    return distances

p = 2
test_pattern = X_scaled[0]
distances = calculate_all_distances(
    test_pattern,
    X_scaled,
    p
)

def create_distance_list(test_pattern, X_train, Y_train, p):
    distance_list = []
    for i in range(len(X_train)):
        distance = minkowski_distance(
            test_pattern,
            X_train[i],
            p
        )
        distance_list.append(
            [i, distance, Y_train[i]]
        )
    return distance_list

distance_list = create_distance_list(
    X_scaled[0],
    X_scaled,
    Y_encoded,
    p=2
)

def bubble_sort(distance_list):
    arr = distance_list.copy()
    n = len(arr)
    for i in range(n - 1):
        for j in range(n - i - 1):
            if (arr[j][1] > arr[j + 1][1] or
                (arr[j][1] == arr[j + 1][1] and
                 arr[j][0] > arr[j + 1][0])):
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

def selection_sort(distance_list):
    arr = distance_list.copy()
    n = len(arr)
    for i in range(n - 1):
        min_index = i
        for j in range(i + 1, n):
            if arr[j][1] < arr[min_index][1]:
                min_index = j
            elif (arr[j][1] == arr[min_index][1] and
                  arr[j][0] < arr[min_index][0]):
                min_index = j
        arr[i], arr[min_index] = arr[min_index], arr[i]
    return arr

def insertion_sort(distance_list):
    arr = distance_list.copy()
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        j = i - 1
        while j >= 0:
            if (arr[j][1] > key[1] or
                (arr[j][1] == key[1] and
                 arr[j][0] > key[0])):
                arr[j + 1] = arr[j]
                j -= 1
            else:
                break
        arr[j + 1] = key
    return arr
    
def sort_distances(distance_list, algorithm):
    if algorithm == "bubble":
        return bubble_sort(distance_list)
    elif algorithm == "selection":
        return selection_sort(distance_list)
    elif algorithm == "insertion":
        return insertion_sort(distance_list)
    else:
        raise ValueError(
            "Invalid sorting algorithm. "
            "Choose bubble, selection, or insertion."
        )

sorting_algorithm = "bubble"
sorted_distances = sort_distances(
    distance_list,
    sorting_algorithm
)


sorted_bubble = bubble_sort(distance_list)
sorted_selection = selection_sort(distance_list)
sorted_insertion = insertion_sort(distance_list)

def identify_neighbours(sorted_distances, k):
    neighbours = sorted_distances[:k]
    return neighbours

k = 5
neighbours = identify_neighbours(
    sorted_distances,
    k
)

def class_evaluation(neighbours):
    class_votes = {}
    for neighbour in neighbours:
        label = neighbour[2]
        distance = neighbour[1]
        if label not in class_votes:
            class_votes[label] = {
                "votes": 0,
                "nearest_distance": distance
            }
        class_votes[label]["votes"] += 1
        if distance < class_votes[label]["nearest_distance"]:
            class_votes[label]["nearest_distance"] = distance
    max_votes = 0
    for label in class_votes:
        if class_votes[label]["votes"] > max_votes:
            max_votes = class_votes[label]["votes"]
    tied_classes = []
    for label in class_votes:
        if class_votes[label]["votes"] == max_votes:
            tied_classes.append(label)
    if len(tied_classes) == 1:
        predicted_class = tied_classes[0]
    else:
        predicted_class = tied_classes[0]
        for label in tied_classes[1:]:
            if (class_votes[label]["nearest_distance"] <
                class_votes[predicted_class]["nearest_distance"]):
                predicted_class = label
    return predicted_class, class_votes

print("Minkowski distance (Euclidean)")
print("Number of distances:", len(distances))
print("First 10 distances:")
print(distances[:10])

print("\nBubble == Selection:", sorted_bubble == sorted_selection)
print("Bubble == Insertion:", sorted_bubble == sorted_insertion)

print("Using:", sorting_algorithm)
for item in sorted_distances[:10]:
    print(item)

print("K =", k)
print("\nNearest neighbours:")
for i, neighbour in enumerate(neighbours, start=1):
    print(
        "Neighbour", i,
        "Index =", neighbour[0],
        "Distance =", neighbour[1],
        "Label =", neighbour[2]
    )

prediction, votes = class_evaluation(neighbours)
print("Class votes:")
print(votes)
print("\nPredicted class:", prediction)
reverse_mapping = {}
for label, code in label_mapping.items():
    reverse_mapping[code] = label
print("Predicted class:",
      reverse_mapping[prediction])

print(df.head())


Number of audio files: 3014
Minkowski distance (Euclidean)
Number of distances: 3014
First 10 distances:
[np.float64(0.0), np.float64(4.432531478888791), np.float64(3.2771456127882215), np.float64(3.2395118916323953), np.float64(4.008687548747024), np.float64(4.285284344123102), np.float64(5.128881593001117), np.float64(5.4644451848553235), np.float64(3.2124142957647637), np.float64(4.4881341208489305)]

Bubble == Selection: True
Bubble == Insertion: True
Using: bubble
[0, np.float64(0.0), 0]
[8, np.float64(3.2124142957647637), 0]
[3, np.float64(3.2395118916323953), 0]
[2, np.float64(3.2771456127882215), 0]
[324, np.float64(3.593546814598648), 0]
[402, np.float64(3.9273341328510347), 0]
[4, np.float64(4.008687548747024), 0]
[448, np.float64(4.274731873389941), 0]
[5, np.float64(4.285284344123102), 0]
[1465, np.float64(4.402431222384479), 0]
K = 5

Nearest neighbours:
Neighbour 1 Index = 0 Distance = 0.0 Label = 0
Neighbour 2 Index = 8 Distance = 3.2124142957647637 Label = 0
Neighbour 3

In [3]:
def calculate_weight(distance):
    if distance == 0:
        return float("inf")
    return 1 / distance
    
def weighted_class_evaluation(neighbours):
    class_weights = {}
    for neighbour in neighbours:
        index = neighbour[0]
        distance = neighbour[1]
        label = neighbour[2]
        weight = calculate_weight(distance)

        if label not in class_weights:
            class_weights[label] = {
                "total_weight": 0,
                "nearest_distance": distance
            }
        class_weights[label]["total_weight"] += weight

        if distance < class_weights[label]["nearest_distance"]:
            class_weights[label]["nearest_distance"] = distance

    for label in class_weights:
        if class_weights[label]["total_weight"] == float("inf"):
            predicted_class = label
            return predicted_class, class_weights
    
    max_weight = -1

    for label in class_weights:
        if class_weights[label]["total_weight"] > max_weight:
            max_weight = class_weights[label]["total_weight"]

    tied_classes = []
    for label in class_weights:
        if class_weights[label]["total_weight"] == max_weight:
            tied_classes.append(label)
            
    if len(tied_classes) == 1:
        predicted_class = tied_classes[0]
    else:
        predicted_class = tied_classes[0]
        for label in tied_classes[1:]:
            if (class_weights[label]["nearest_distance"] <
                class_weights[predicted_class]["nearest_distance"]):
                predicted_class = label
                
    return predicted_class, class_weights


k = 5
p = 2
sorting_algorithm = "bubble"

test_pattern = X_scaled[0]

distance_list = create_distance_list(
    test_pattern,
    X_scaled,
    Y_encoded,
    p
)

sorted_distances = sort_distances(
    distance_list,
    sorting_algorithm
)

neighbours = identify_neighbours(
    sorted_distances,
    k
)

prediction, weighted_votes = weighted_class_evaluation(
    neighbours
)

print("K =", k)
print("p =", p)
print("Sorting algorithm =", sorting_algorithm)

print("\nK Neighbours:")
for neighbour in neighbours:
    print(neighbour)

print("\nWeighted class evaluation:")
print(weighted_votes)

print("\nPredicted encoded class:", prediction)

K = 5
p = 2
Sorting algorithm = bubble

K Neighbours:
[0, np.float64(0.0), 0]
[8, np.float64(3.2124142957647637), 0]
[3, np.float64(3.2395118916323953), 0]
[2, np.float64(3.2771456127882215), 0]
[324, np.float64(3.593546814598648), 0]

Weighted class evaluation:
{0: {'total_weight': np.float64(inf), 'nearest_distance': np.float64(0.0)}}

Predicted encoded class: 0


In [4]:
from sklearn.model_selection import train_test_split
import numpy as np

Y_encoded = np.array(Y_encoded)
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=Y_encoded
)

print("Training data shape:", X_train.shape)
print("Training labels shape:", Y_train.shape)

print("Testing data shape:", X_test.shape)
print("Testing labels shape:", Y_test.shape)

print("\nTraining class distribution:")
print(np.bincount(Y_train))

print("\nTesting class distribution:")
print(np.bincount(Y_test))

Training data shape: (2411, 26)
Training labels shape: (2411,)
Testing data shape: (603, 26)
Testing labels shape: (603,)

Training class distribution:
[1205 1206]

Testing class distribution:
[302 301]


In [5]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

def standardize_train_test(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    return X_train_scaled, X_test_scaled, mean, std

X_train_scaled, X_test_scaled, train_mean, train_std = standardize_train_test(
    X_train,
    X_test
)

neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train_scaled, Y_train)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("KNN classifier trained successfully.")

X_train_scaled shape: (2411, 26)
X_test_scaled shape: (603, 26)
KNN classifier trained successfully.


In [6]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=Y_encoded
)

def standardize_train_test(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled, mean, std

X_train_scaled, X_test_scaled, train_mean, train_std = standardize_train_test(
    X_train,
    X_test
)

neigh = KNeighborsClassifier(n_neighbors=3)

neigh.fit(X_train_scaled, Y_train)

accuracy = neigh.score(X_test_scaled, Y_test)
prediction = neigh.predict(X_test_scaled)

print("KNN Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)
print("Prediction:", prediction)

KNN Accuracy: 0.9900497512437811
Accuracy (%): 99.00497512437812
Prediction: [0 0 0 0 0 1 0 0 1 0 0 1 1 1 1 1 0 1 0 1 1 1 1 0 1 1 1 0 0 0 1 0 0 1 0 1 0
 1 1 1 0 1 1 1 1 1 0 1 0 0 0 1 0 0 1 1 0 1 1 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0
 1 0 0 0 0 1 1 1 0 0 0 1 0 0 1 0 1 1 1 1 1 1 0 0 1 0 0 1 1 1 1 1 1 0 1 0 0
 1 0 0 1 0 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 0 0 0 1 0 1 1 1 0 0 0 1 0 1 1
 0 0 1 0 1 0 0 1 0 0 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 0 0 1 0 0 1 1 1 0 1 0 1
 0 1 1 0 0 1 0 0 0 0 1 0 1 0 0 1 0 1 0 1 1 1 1 0 1 1 0 1 1 1 0 0 1 0 0 1 1
 1 1 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 0 1
 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0 1
 1 0 0 0 1 0 1 1 0 1 1 0 1 1 1 0 1 1 0 1 1 1 1 0 0 0 0 1 0 1 0 1 0 1 0 0 0
 1 1 0 0 0 1 1 0 0 1 1 0 1 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 0 1 1 1 0 1 1 0 0
 1 0 0 0 1 1 0 1 0 1 1 1 0 1 1 0 1 0 0 1 0 1 1 0 0 1 1 1 1 1 0 0 0 0 0 0 1
 0 0 1 1 0 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1 0 1 0 1 0 0 0 0
 1 1 1 1 1 0 1 1 1 1 1 

In [8]:
import numpy as np

class MyKNN:

    def __init__(self, k=3, p=2, sorting_algorithm="bubble"):    
        self.k = k
        self.p = p
        self.sorting_algorithm = sorting_algorithm
        self.X_train = None
        self.Y_train = None

    def fit(self, X_train, Y_train):
        self.X_train = np.array(X_train)
        self.Y_train = np.array(Y_train)
        return self

    def minkowski_distance(self, X, Y):
        total = 0
        for i in range(len(X)):
            total += abs(X[i] - Y[i]) ** self.p
        return total ** (1 / self.p)

    def calculate_distances(self, test_pattern):
        distances = []
        for i in range(len(self.X_train)):
            distance = self.minkowski_distance(
                test_pattern,
                self.X_train[i]
            )
            distances.append(
                [i, distance, self.Y_train[i]]
            )

        return distances

    def bubble_sort(self, distances):
        distances = distances.copy()
        n = len(distances)
        for i in range(n):
            for j in range(0, n - i - 1):
                current = distances[j]
                next_item = distances[j + 1]
                if current[1] > next_item[1]:
                    distances[j], distances[j + 1] = \
                        distances[j + 1], distances[j]
                elif (current[1] == next_item[1] and
                      current[0] > next_item[0]):
                    distances[j], distances[j + 1] = \
                        distances[j + 1], distances[j]
        return distances

    def selection_sort(self, distances):
        distances = distances.copy()
        n = len(distances)

        for i in range(n):

            min_index = i

            for j in range(i + 1, n):

                if distances[j][1] < distances[min_index][1]:

                    min_index = j

                elif (distances[j][1] == distances[min_index][1]
                      and distances[j][0] < distances[min_index][0]):

                    min_index = j

            distances[i], distances[min_index] = \
                distances[min_index], distances[i]

        return distances

    def insertion_sort(self, distances):

        distances = distances.copy()

        for i in range(1, len(distances)):

            current = distances[i]

            j = i - 1

            while j >= 0:

                should_shift = False

                if distances[j][1] > current[1]:

                    should_shift = True

                elif (distances[j][1] == current[1]
                      and distances[j][0] > current[0]):

                    should_shift = True

                if should_shift:

                    distances[j + 1] = distances[j]
                    j -= 1

                else:
                    break

            distances[j + 1] = current

        return distances

    def sort_distances(self, distances):

        if self.sorting_algorithm == "bubble":

            return self.bubble_sort(distances)

        elif self.sorting_algorithm == "selection":

            return self.selection_sort(distances)

        elif self.sorting_algorithm == "insertion":

            return self.insertion_sort(distances)

        else:

            raise ValueError(
                "Invalid sorting algorithm. "
                "Use bubble, selection or insertion."
            )

    def identify_neighbours(self, sorted_distances):

        return sorted_distances[:self.k]

    def evaluate_class(self, neighbours):

        class_votes = {}

        for neighbour in neighbours:

            label = neighbour[2]

            if label not in class_votes:

                class_votes[label] = 0

            class_votes[label] += 1


        # Find maximum vote
        max_votes = max(class_votes.values())


        # Classes having maximum votes
        tied_classes = []

        for label in class_votes:

            if class_votes[label] == max_votes:

                tied_classes.append(label)

        if len(tied_classes) == 1:

            return tied_classes[0]


        for neighbour in neighbours:

            if neighbour[2] in tied_classes:

                return neighbour[2]

    def predict_one(self, test_pattern):

        distances = self.calculate_distances(
            test_pattern
        )

        sorted_distances = self.sort_distances(
            distances
        )

        neighbours = self.identify_neighbours(
            sorted_distances
        )

        prediction = self.evaluate_class(
            neighbours
        )

        return prediction

    def predict(self, X_test):

        predictions = []

        for test_pattern in X_test:

            prediction = self.predict_one(
                test_pattern
            )

            predictions.append(prediction)

        return np.array(predictions)

    def score(self, X_test, Y_test):

        predictions = self.predict(X_test)

        Y_test = np.array(Y_test)

        correct = 0

        for i in range(len(Y_test)):

            if predictions[i] == Y_test[i]:

                correct += 1

        accuracy = correct / len(Y_test)

        return accuracy

knn = MyKNN(
    k=3,
    p=2,
    sorting_algorithm="bubble"
)

knn.fit(
    X_train_scaled,
    Y_train
)

Y_pred = knn.predict(
    X_test_scaled
)

print("Predictions:")
print(Y_pred[:20])

accuracy = knn.score(
    X_test_scaled,
    Y_test
)

print("Custom KNN Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

print("KNN model fitted successfully.")

KeyboardInterrupt: 